In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "8GWiqzepPUZc"
   },
   "source": [
    "# Model Fine-Tuning with Unsloth"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "Iz3InP0nU6Nx"
   },
   "source": [
    "Welcome!\n",
    "\n",
    "This recipe provides a guide to **fine-tuning Granite 3.3 2B Instruct model** using [Unsloth](https://github.com/unslothai/unsloth?tab=readme-ov-file), an optimized open-source framework designed for efficient LLM fine-tuning and reinforcement learning.\n",
    "\n",
    "Fine-tuning refers to the process of further training a pre-trained model on a task-specific dataset to improve its performance in specialized contexts. Here, we focus on Low-Rank Adaptation (LoRA), a method within the broader category of Parameter-Efficient Fine-Tuning (PEFT), where only a subset of model parameters are modified. PEFT methods preserve the majority of the pre-trained knowledge, hence minimizing the risks of catastrophic forgetting.\n",
    "\n",
    "Fine-tuning is particularly valuable when prompting or retrieval-based techniques fall short. While prompt engineering enables zero-shot or few-shot learning, it often results in inconsistent outputs, especially for complex tasks or domain-specific requirements. Similarly, Retrieval-Augmented Generation (RAG) enhances factual grounding by incorporating external context but does not alter the model’s underlying reasoning or stylistic behavior. Fine-tuning addresses these limitations by embedding the desired patterns, tone, and logic directly into the model, resulting in more robust and reliable outputs.\n",
    "\n",
    "There are several distinct types of fine-tuning, each suited to different use cases. Instruction tuning aligns the model with general task-following behavior, conversation tuning optimizes it for dialogue and multi-turn interactions, and domain-specific tuning adapts the model to specialized fields.\n",
    "\n",
    "This recipe explores domain specific training and fine-tunes the model to perform better on math reasoning tasks.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "c8y5bG8jTroB"
   },
   "source": [
    "## Colab Notes"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "1X6SSe5F5p5R"
   },
   "source": [
    "\n",
    "**THIS NOTEBOOK WORKS IN LINUX/WINDOWS ENVIRONMENT AND REQUIRES A CUDA-ENABLED GPU (NVIDIA GPU).**\n",
    "\n",
    "Please refer to the Unsloth system requirements [here](https://docs.unsloth.ai/get-started/beginner-start-here/unsloth-requirements).\n",
    "\n",
    "This notebook is designed to work with the free level of GPU available from Google Colab, the T4 GPU. You should not need to pay for GPU time to run this simple fine-tuning demo.\n",
    "\n",
    "If you want to fine-tune using larger datasets, you will need a machine or runtime with a large GPU to perform tuning. Your local computer can't run this notebook without a CUDA-enabled GPU.\n",
    "\n",
    "**Troubleshooting:**\n",
    "\n",
    "- **Verify runtime type**: Under the \"Runtime\" menu, select \"Change runtime type.\" The \"Hardware accelerator\" field must be set to T4 GPU.\n",
    "- **Verify runtime is connected**: In the top right, you should see the connection status for the T4 runtime. You should see a green checkmark, next to \"T4\", and \"RAM Disk.\".\n",
    "- **Verify T4 GPU is only connected to one Colab session**: If you see the word \"Connecting\" in the top right and it doesn't seem to be resolving, click the arrow dropdown next to it and choose \"Manage sessions\". If there is an active session already (say, from another run of the notebook in a different browser window, or from using another notebook), you will need to disconnect the session. Click \"Terminate other sessions\" to do so."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "CrRNXzt8ufge"
   },
   "source": [
    "## Install Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "wIH_Yt1aTGjs",
    "outputId": "acc3e8a2-86a3-47d7-91e0-eaa2bd30d471"
   },
   "outputs": [],
   "source": [
    "%pip install git+https://github.com/ibm-granite-community/utils \\\n",
    "  sentencepiece protobuf \"datasets>=3.4.1,<4.0.0\" \"huggingface_hub>=0.34.0\" hf_transfer\n",
    "\n",
    "%pip install --no-deps bitsandbytes \\\n",
    "  accelerate \\\n",
    "  xformers==0.0.29.post3 \\\n",
    "  peft \\\n",
    "  trl \\\n",
    "  tqdm \\\n",
    "  triton \\\n",
    "  cut_cross_entropy \\\n",
    "  unsloth_zoo \\\n",
    "  unsloth"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "cedAoOdtvJai"
   },
   "source": [
    "## Fine Tuning Granite Model"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "k6HuHjFGczMU"
   },
   "source": [
    "The Granite 3.3 2B model, while generally proficient in natural language understanding and generation, may struggle when tasked with specialized reasoning challenges such as high-accuracy mathematical problem solving. These limitations are expected, given the relatively smaller parameter size of the model. To address this, fine-tuning the Granite 3.3 2B model on domain-specific mathematical dataset is a promising approach to improve its accuracy and reliability in quantitative tasks."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "KShI9NiiHfEW"
   },
   "source": [
    "### Loading the base model"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "8gkcj-BHy3zH"
   },
   "source": [
    "In this section, we load the Granite 3.3 2B Instruct base model, preparing it for fine-tuning."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 233,
     "referenced_widgets": [
      "49c93f8fe511494393e5b76c9364c84d",
      "18f0b4855a864a28943caf933c1ba1a1",
      "30f6d2b46e2f4635a2a62b60ed708dbd",
      "2fba2af1219048c19a709ee3b2a1d62c",
      "02770933e27444fda3be7ccd00d49ae7",
      "1d39fac4e6fb46739a6f58ad17a82212",
      "58058a82b4584f20860929147c39f838",
      "cfd476b3adfc453f9ae041d1f66d898a",
      "bb1e40cf2e484e11875ab78fea174503",
      "d02ba1ec2f3547d8b3a3ffefb8c6345b",
      "9be55f4c4b4048dda6c4983cb26fb90a"
     ]
    },
    "id": "QdgxJuA5u_Iq",
    "outputId": "1924473d-df88-409c-e4a3-d2d5f8523486"
   },
   "outputs": [],
   "source": [
    "from unsloth import FastLanguageModel\n",
    "\n",
    "base_model, tokenizer = FastLanguageModel.from_pretrained(\n",
    "    model_name=\"ibm-granite/granite-3.3-2b-instruct\",\n",
    "    max_seq_length = 2048,\n",
    "    dtype = None,\n",
    "    load_in_4bit = False\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "OaTq78_rbZKh"
   },
   "source": [
    "### Prepare the Math Dataset"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "HbFRUNkszVDB"
   },
   "source": [
    "Here, the code formats a math dataset into a chat-style prompt-response structure using the tokenizer's chat template."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "xfruESI6Q4q2"
   },
   "outputs": [],
   "source": [
    "EOS_TOKEN = tokenizer.eos_token\n",
    "def formatting_prompts_func(examples):\n",
    "    messages = []\n",
    "    for i in range(len(examples[\"problem\"])):\n",
    "        messages.append([\n",
    "            {\"role\": \"user\", \"content\": examples[\"problem\"][i]},\n",
    "            {\"role\": \"assistant\", \"content\": examples[\"solution\"][i]}\n",
    "        ])\n",
    "    texts = [tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = False) + EOS_TOKEN for message in messages]\n",
    "    return { \"text\" : texts, }"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "goA-dO8yQ8SX"
   },
   "outputs": [],
   "source": [
    "from datasets import load_dataset\n",
    "\n",
    "dataset = load_dataset(\"xDAN2099/lighteval-MATH\", split=\"train[:500]\", trust_remote_code=True)\n",
    "dataset = dataset.map(\n",
    "    formatting_prompts_func,\n",
    "    batched=True,\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "BvZ7pMYP19ZI"
   },
   "source": [
    "This is how the final fine-tuning dataset samples looks like:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "OEWh-2p3RBAG",
    "outputId": "b81a6962-225e-4830-cd0a-adadd38e1947"
   },
   "outputs": [],
   "source": [
    "print(dataset[\"text\"][0])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "O6ew0rqTyprA"
   },
   "source": [
    "### LoRA fine tuning"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "B15d-yXI2b3g"
   },
   "source": [
    "We now add LoRA adapters for parameter efficient finetuning."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "4OWnHgDbyf8e",
    "outputId": "27fde165-9e6f-433b-97e8-103a1e34d6b5"
   },
   "outputs": [],
   "source": [
    "target_modules =  [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\",\n",
    "                   \"gate_proj\", \"up_proj\", \"down_proj\"]\n",
    "\n",
    "model = FastLanguageModel.get_peft_model(\n",
    "    base_model,\n",
    "    r = 16, # Rank of lora matrices\n",
    "    target_modules = target_modules,  # Modules of the llm the lora weights are used\n",
    "    lora_alpha = 16, # scales the weights of the adapters\n",
    "    lora_dropout = 0, # Unsloth recommends 0 is better for fast patching\n",
    "    bias = \"none\",    # \"none\" is optimized\n",
    "    use_gradient_checkpointing = \"unsloth\", #\"unsloth\" for very long context, decreases vram\n",
    "    random_state = 3407,\n",
    "    use_rslora = False,\n",
    "    loftq_config = None,\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "9pVUQTDd3urM"
   },
   "source": [
    "Using Hugging Face TRL's SFTTrainer, we configure the training environment for the base model. Feel free to experiment with the training arguments to observe their impact on model performance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "Rv-FAiRHyt1n",
    "outputId": "f1a355ff-c888-4be7-ccb3-793be309d061"
   },
   "outputs": [],
   "source": [
    "from trl import SFTTrainer\n",
    "from transformers import TrainingArguments\n",
    "from unsloth import is_bfloat16_supported\n",
    "\n",
    "trainer = SFTTrainer(\n",
    "    model = model,\n",
    "    tokenizer = tokenizer,\n",
    "    train_dataset = dataset,\n",
    "    dataset_text_field = \"text\",\n",
    "    max_seq_length = 2048,\n",
    "    dataset_num_proc = 2,\n",
    "    args = TrainingArguments(\n",
    "        per_device_train_batch_size = 2,\n",
    "        gradient_accumulation_steps = 4,\n",
    "        num_train_epochs = 2,\n",
    "        learning_rate = 2e-4,\n",
    "        fp16 = not is_bfloat16_supported(),\n",
    "        bf16 = is_bfloat16_supported(),\n",
    "        logging_steps = 25,\n",
    "        optim = \"adamw_8bit\",\n",
    "        weight_decay = 0.01,\n",
    "        lr_scheduler_type = \"linear\",\n",
    "        seed = 3407,\n",
    "        output_dir = \"outputs\",\n",
    "        report_to = \"none\"\n",
    "    ),\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "48SoOXw05VMm"
   },
   "source": [
    "We now initiate the fine-tuning of the model. With current training environment, the 1.1% of the parameters are trainable and it takes ~7 mins with the specified training arguments."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 363
    },
    "id": "ekUtiAz2y17y",
    "outputId": "d089a7b7-6206-45a8-aad5-3c5c2270bd5a"
   },
   "outputs": [],
   "source": [
    "trainer_stats = trainer.train()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "lPlx1Mmk8vO1"
   },
   "source": [
    "Here are some training statistics for your reference:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "2_EREU6R5gYZ",
    "outputId": "4e3854d1-dbe5-4848-f5d0-a2093800a168"
   },
   "outputs": [],
   "source": [
    "trainer_stats"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "kMUe5Lr3y_q6"
   },
   "source": [
    "## Inference the fine-tuned model"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "hHZt85Fx7-QM"
   },
   "source": [
    "The fine-tuned model is ready for inferencing!"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "0vEk-BoXcSYa",
    "outputId": "489fa9f4-203d-431f-b8d1-a745569da7ea"
   },
   "outputs": [],
   "source": [
    "from ibm_granite_community.notebook_utils import wrap_text\n",
    "\n",
    "FastLanguageModel.for_inference(model)\n",
    "\n",
    "if tokenizer.pad_token is None:\n",
    "    tokenizer.pad_token = tokenizer.eos_token\n",
    "\n",
    "query = \"If $x = 2$ and $y = 5$, then what is the value of $\\frac{x^4+2y^2}{6}$ ?\"\n",
    "\n",
    "messages = [\n",
    "    {\"role\": \"user\", \"content\": query}\n",
    "]\n",
    "\n",
    "# Tokenize and return both input_ids and attention_mask\n",
    "encoding = tokenizer.apply_chat_template(\n",
    "    messages,\n",
    "    tokenize=True,\n",
    "    add_generation_prompt=True,\n",
    "    return_tensors=\"pt\",\n",
    "    return_dict=True\n",
    ").to(\"cuda\")\n",
    "\n",
    "# Generate using both input_ids and attention_mask\n",
    "output_ids = model.generate(\n",
    "    encoding[\"input_ids\"],\n",
    "    attention_mask=encoding[\"attention_mask\"],\n",
    "    pad_token_id=tokenizer.eos_token_id,\n",
    "    max_new_tokens=2560,\n",
    "    use_cache=True\n",
    ")\n",
    "\n",
    "# Decode only the newly generated tokens\n",
    "response = tokenizer.decode(\n",
    "    output_ids[0][encoding[\"input_ids\"].shape[-1]:],\n",
    "    skip_special_tokens=True\n",
    ")\n",
    "print(wrap_text(response))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "jJTGMwhcaj6X"
   },
   "source": [
    "The expected response must be along the lines of:\n",
    "\n",
    "\n",
    "> We have  \\[\\frac{x^4 + 2y^2}{6} = \\frac{2^4 + 2(5^2)}{6} = \\frac{16+2(25)}{6} = \\frac{16+50}{6} = \\frac{66}{6} = \\boxed{11}.\\]\n",
    "\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "F_9numO39Vti"
   },
   "source": [
    "You can also use a [TextStreamer](https://huggingface.co/docs/transformers.js/en/api/generation/streamers#module_generation/streamers.TextStreamer) for real-time inference, allowing you to view the model’s output token by token as it’s generated, rather than waiting for the full response. This is demonstrated in the next section."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "9b6Z-zbYWp21"
   },
   "source": [
    "## Saving Fine Tuned model"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "D7psby_mWt1V"
   },
   "source": [
    "The fine-tuned models can either be saved locally or online on HuggingFace. The models can then be loaded using FastLanguage model and set for inference."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "upcOlWe7A1vc",
    "outputId": "bc63fda7-ed73-4fd9-e311-e5053e79f284"
   },
   "outputs": [],
   "source": [
    "model.save_pretrained(\"Finetuned_Granite\")  # Local saving\n",
    "tokenizer.save_pretrained(\"Finetuned_Granite\") # Local saving\n",
    "# model.push_to_hub(\"your_name/lora_model\", token = \"...\") # Online saving\n",
    "# tokenizer.push_to_hub(\"your_name/lora_model\", token = \"...\") # Online saving"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 673,
     "referenced_widgets": [
      "bcbc4c662dcc4ab0b2159199b3e6d490",
      "ca7b75cde110497da0ad63eb8971e1a9",
      "b3e5501a03fe4779b6a8cf2a5bcf7f1c",
      "4fa2575e9a474a9687d7f8060bd93b09",
      "96702a6e131a4576a53f3d0e7826c994",
      "8a07bd73244845e7951a68c79895bfe2",
      "883a234d8682446597fefc61608e9433",
      "01eeb9510e6c4816a7c01ca2cb53a3da",
      "c2e952d4cf2544c2932227bed12b4c3d",
      "9927a3e34197434887d7f65ca8483b8d",
      "1f9828ac127d44e1b4fc1af77762fe78"
     ]
    },
    "id": "ukhInbmja6nj",
    "outputId": "bcbbf097-1e74-42dc-dede-c57edbf0805e"
   },
   "outputs": [],
   "source": [
    "from unsloth import FastLanguageModel\n",
    "model, tokenizer = FastLanguageModel.from_pretrained(\n",
    "    model_name = \"Finetuned_Granite\", # Locally saved model\n",
    "    max_seq_length = 2048,\n",
    "    dtype = None,\n",
    "    load_in_4bit = False,\n",
    ")\n",
    "FastLanguageModel.for_inference(model)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "VIbfewS7cqkR",
    "outputId": "4aed49ea-3a12-4d18-9314-cfdb1bbf3215"
   },
   "outputs": [],
   "source": [
    "query = \"Simplify $\\sqrt[3]{1+8} \\cdot \\sqrt[3]{1+\\sqrt[3]{8}}$.\"\n",
    "messages = [\n",
    "    {\"role\": \"user\", \"content\": query},\n",
    "]\n",
    "inputs = tokenizer.apply_chat_template(\n",
    "    messages,\n",
    "    tokenize = True,\n",
    "    add_generation_prompt = True, # Must add for generation\n",
    "    return_tensors = \"pt\",\n",
    ").to(\"cuda\")\n",
    "\n",
    "# Create attention mask\n",
    "attention_mask = (inputs != tokenizer.pad_token_id).long()\n",
    "\n",
    "# Generate output with attention mask\n",
    "from transformers import TextStreamer\n",
    "text_streamer = TextStreamer(tokenizer, skip_prompt=True)\n",
    "\n",
    "_ = model.generate(\n",
    "    input_ids=inputs,\n",
    "    attention_mask=attention_mask,\n",
    "    streamer=text_streamer,\n",
    "    max_new_tokens=256,\n",
    "    use_cache=True,\n",
    "    temperature=1.5,\n",
    "    min_p=0.1\n",
    ")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "oUUnuCqScyFb"
   },
   "source": [
    "Expected response:\n",
    "\n",
    "\n",
    ">The first cube root becomes $\\sqrt[3]{9}$. $\\sqrt[3]{8}=2$, so the second cube root becomes $\\sqrt[3]{3}$. Multiplying these gives $\\sqrt[3]{27} = \\boxed{3}$.\n",
    "\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "7InuZDaadEN8"
   },
   "source": [
    "## Conclusion"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "KFFVSibRdi0O"
   },
   "source": [
    "This recipe demonstrated how to fine-tune the Granite 3.3 2B model using LoRA with Unsloth. It also showcased the complete workflow for saving the fine-tuned model locally and performing inference."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "hQTNQm4ueOyr"
   },
   "source": [
    "## References"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "3It4mraYeQ9h"
   },
   "source": [
    "1. [Fine Tuning - IBM Think Blog](https://www.ibm.com/think/topics/fine-tuning)\n",
    "2. [Unsloth Docs](https://docs.unsloth.ai/)"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "gpuType": "T4",
   "provenance": []
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}